# Dask Bag (2)

__Автор задач: Блохин Н.В. (NVBlokhin@fa.ru)__

Материалы:
* Макрушин С.В. Лекция "Map-Reduce"
* https://docs.dask.org/en/latest/bag.html
* Jesse C. Daniel. Data Science with Python and Dask.
* https://realpython.com/python-reduce-function/
* https://docs.python.org/3/library/calendar.html

## Задачи для совместного разбора

In [ ]:
import dask.bag as db
import json
from functools import reduce

In [ ]:
with open("./data/posts.json") as fp:
    posts = json.load(fp)
posts_bag = db.from_sequence(posts, npartitions=2)

1. Посчитайте суммарную длину длин описаний постов

In [ ]:
sum(len(p['body']) for p in posts_bag)

16064

![image-2.png](attachment:image-2.png)

In [ ]:
def s(acc, val):
    return acc + val

reduce(
    s,
    map(
        lambda p: len(p['body']),
        posts_bag
    ),
    0
)

16064

![image.png](attachment:image.png)

In [ ]:
body = posts_bag.map(lambda p: len(p['body']))
body.map_partitions(
    lambda parttion: (sum(parttion), )
).sum().compute()

16064

![image-3.png](attachment:image-3.png)

In [ ]:
body.reduction(
    lambda partition: sum(partition),
    lambda partition_results: sum(partition_results)
).compute()

16064

![image-2.png](attachment:image-2.png)

In [ ]:
def binop(acc, val):
    return acc + val

def combine(acc, val):
    return acc + val

body.fold(
    binop=binop,
    combine=combine,
).compute()

16064

2. При помощи метода `foldby` посчитайте, сколько постов написал каждый из пользователей.

![image-4.png](attachment:image-4.png)

In [ ]:
posts_bag.foldby(
    key=lambda p: p['userId'],
    binop=lambda acc, val: acc + 1,
    initial=0,
    combine=lambda acc, val: acc + val,
    combine_initial=0,
).compute()

[(1, 10),
 (2, 10),
 (3, 10),
 (4, 10),
 (5, 10),
 (6, 10),
 (7, 10),
 (8, 10),
 (9, 10),
 (10, 10)]

## Лабораторная работа 10

__При решении данных задач не подразумевается использования циклов или генераторов Python в ходе работы с пакетами `numpy`, `pandas` и `dask`, если в задании не сказано обратного. Решения задач, в которых для обработки массивов `numpy`, структур `pandas` или структур `dask` используются явные циклы (без согласования с преподавателем), могут быть признаны некорректными и не засчитаны.__

В ходе выполнения все операции вычислений проводятся над `dask.bag` и средствами пакета `dask`, если в задании не сказано обратного. Переход от `dask.bag` к любым другим структурам возможен исключительно для демонстрации результата в конце решения задачи. Если в задаче используются результаты выполнения предыдущих задач, то подразумевается, что вы используете результаты в виде `dask.bag` (то есть то, что было получено до вызова `compute`, а не после).

<p class="task" id="1"></p>

1\. Загрузите отзывы из файла `reviews_1.json` в виде списка. Используя `functools.reduce`, сгенерируйте словарь, содержащий частоты слов, встречающихся в отзывах из этого файла. Для разбиения на слова используйте объект `tokenizer`. Перед разбиением на слова приведите текст отзыва к нижнему регистру. Выведите на экран длину полученного словаря.


In [ ]:
from functools import reduce
from nltk.tokenize import RegexpTokenizer
from collections import Counter
tokenizer = RegexpTokenizer(r'\w+')

In [ ]:
with open("./data/reviews_1.json") as fp:
    lines = fp.readlines()
reviews = [json.loads(line) for line in lines]
reviews = [review for review in reviews if review['review'] is not None]

def sum_dicts(d1, d2):
    return {word: d1.get(word, 0) + d2.get(word, 0) for word in set(d1) | set(d2)}

def s(acc, val):
    words = tokenizer.tokenize(val['review'].lower())
    return sum_dicts(acc, Counter(words))

words_dict = reduce(
    s,
    reviews,
    {}
)

len(words_dict)

41570

<p class="task" id="2"></p>

2\. Общее количество символов, которое занимает слово во всех отзывах, можно рассчитать, умножив длину этого слова на частоту использования этого слова. Используя `functools.reduce`, найдите слово, которое занимает больше всего символов в отзывах из файла `reviews_1.json` (воспользуйтесь результатами из задачи 1). Выведите найденное слово и количество занимаемых им символов на экран. Решите ту же задачу при помощи функции `max`.

In [ ]:
def s(acc, val):
    if len(val[0]) * val[1] > len(acc[0]) * acc[1]:
        return val
    return acc

res = reduce(
    s,
    words_dict.items(),
    ('', 0)
)
res[0], res[1] * len(res[0])

('the', 711756)

In [ ]:
res = max(words_dict.items(), key=lambda x: len(x[0]) * x[1])
res[0], res[1] * len(res[0])

('the', 711756)

<p class="task" id="3"></p>

3\. Будем считать, что сегмент _плохо перемешан_, если в нем _подряд_ идет 5 или более отзывов, оставленных в один и тот же год. Воспользовавшись методом `Bag.map_partitions`, посчитайте и выведите на экран, сколько сегментов оказались _плохо перемешанными_ в `reviews_bag`.

In [ ]:
import dask.bag as db
import json
# Перед выполнением убедитесь, что у вас существует папка data/reviews_full
reviews_bag = db.read_text(
    "data/reviews_full/*.json", blocksize="128MiB"
).map(json.loads)

In [ ]:
def segment_is_bad(segment):
    years = [review['date'][:4] for review in segment]
    for i in range(len(years) - 4):
        if years[i] == years[i + 1] == years[i + 2] == years[i + 3] == years[i + 4]:
            return True
    return False

reviews_bag.map_partitions(
    lambda partition: (segment_is_bad(partition), )
).sum().compute()

17

<p class="task" id="4"></p>

4\. Будем считать, что сегмент _плохо перемешан_, если в нем подряд идет 5 или более отзывов, оставленных в один и тот же год. Воспользовавшись методом `Bag.reduction`, посчитайте и выведите на экран, сколько сегментов оказались _плохо перемешанными_ в `reviews_bag`.

In [ ]:
reviews_bag.reduction(
    segment_is_bad,
    lambda partition_results: sum(partition_results)
).compute()

17

<p class="task" id="5"></p>

5\. Будем считать, что сегмент _плохо перемешан_, если в нем подряд идет 5 или более отзывов, оставленных в один и тот же год. Воспользовавшись методом `fold`, посчитайте и выведите на экран, сколько сегментов оказались _плохо перемешанными_ в `reviews_bag`.

Примечание 1: один из возможных вариантов реализации функции свертки заключается в использовании кортежей, содержащих год и кол-во раз, которое этот год встретился подряд.

Примечание 2: dask автоматически объединяет сегменты в группы размера `split_every`. Из-за этого во время шага `combine` могут возникнуть проблемы с типом входных данных. Вы можете упростить функцию `combine`, выставив достаточно большое значение `split_every`.

In [ ]:
def binop(acc, val):
    if acc[1] > 4:
        return acc
    if acc[0] == val['date'][:4]:
        return (acc[0], acc[1] + 1)
    return (val['date'][:4], 1)

def combine(acc, val):
    if type(acc) == tuple:
        if acc[1] > 4:
            acc = 1
        else:
            acc = 0
    if val[1] > 4:
        acc += 1
    return acc

reviews_bag.fold(
    binop=binop,
    combine=combine,
    initial=('', 0),
    split_every=1000,
).compute()

17

<p class="task" id="6"></p>

6\.  При помощи метода `accumulate` для каждого $i$-го отзыва расчитайте, столько отзывов из 2010 года было оставлено среди первых $i$ элементов. Воспользовавшись полученным результатом и расчитав общее количество элементов в `reviews_bag`, выберите `k` первых строк объекта `reviews_bag` таким образом, чтобы в выбранном множестве оказалось ровно 10 тыс. отзывов, оставленных в 2010 году. Подтвердите правильность решения, выведя количество элементов в выбранном множестве, которые были оставлены в 2010 году.

In [ ]:
def binop(acc, val):
    if val['date'][:4] == '2010':
        return acc + 1
    return acc

reviews_2010_accumulate = reviews_bag.accumulate(
    binop=binop,
    initial=0,
)

In [ ]:
reviews_2010_accumulate.take(244200)[-1]

10000

<p class="task" id="7"></p>

7\. Посчитайте, сколько отзывов оставили пользователи в каждом месяце каждого года. Создайте `pd.DataFrame`, у которого в качестве индексов строк указаны года, а в качестве имен столбцов - названия месяцев. Выведите полученную таблицу на экран. Выведите на экран строку за 2010 год.

Для преобразования числа месяца в название вы можете воспользоваться пакетом [calendar](https://docs.python.org/3/library/calendar.html#calendar.month_name)

In [ ]:
import calendar
import pandas as pd

def binop(acc, val):
    year, month = val['date'][:4], val['date'][5:7]
    if year not in acc:
        acc[year] = {}
    if month not in acc[year]:
        acc[year][month] = 1
    else:
        acc[year][month] += 1
    return acc

def combine(acc, val):
    for year, months in val.items():
        if year not in acc:
            acc[year] = {}
        for month, count in months.items():
            if month not in acc[year]:
                acc[year][month] = count
            else:
                acc[year][month] += count
    return acc

reviews_fold = reviews_bag.fold(
    binop=binop,
    combine=combine,
    initial={},
    split_every=1000,
).compute()

In [ ]:
reviews_df = pd.DataFrame(reviews_fold).T.sort_index(axis=0).sort_index(axis=1)
reviews_df.columns = [calendar.month_name[i] for i in range(1, 13)]
reviews_df.loc['2010']

January      35728.0
February     30675.0
March        32592.0
April        32791.0
May          32855.0
June         33614.0
July         30057.0
August       28009.0
September    26905.0
October      28239.0
November     27878.0
December     29077.0
Name: 2010, dtype: float64

<p class="task" id="8"></p>

8\. Используя метод `Bag.foldby`, подсчитайте и выведите на экран максимальную длину отзывов в зависимости от года в объекте `reviews_bag`.

In [ ]:
reviews_bag.foldby(
    key=lambda review: review['date'][:4],
    binop=lambda acc, val: max(acc, len(val['review'])) if val['review'] is not None else acc,
    initial=0,
    combine=lambda acc, val: max(acc, val),
    combine_initial=0,
).compute()

[('2016', 4954),
 ('2006', 5567),
 ('1985', 3937),
 ('2019', 6972),
 ('1972', 2036),
 ('2014', 6972),
 ('2017', 5567),
 ('1978', 3717),
 ('2015', 6972),
 ('2009', 5567),
 ('1989', 4289),
 ('2004', 4566),
 ('2020', 6972),
 ('2010', 5567),
 ('2008', 6972),
 ('2013', 6972),
 ('2012', 8587),
 ('2021', 5799),
 ('1997', 4521),
 ('2018', 6972),
 ('2005', 8587),
 ('2000', 5799),
 ('2011', 4592),
 ('1994', 4289),
 ('1998', 4289),
 ('2001', 3535),
 ('2003', 5567),
 ('1980', 3219),
 ('1984', 4521),
 ('1992', 4521),
 ('2007', 5799),
 ('2002', 6972),
 ('1993', 6972),
 ('1990', 4521),
 ('1999', 3535),
 ('1977', 4396),
 ('1982', 5799),
 ('1995', 5799),
 ('1991', 4146),
 ('1988', 3535),
 ('1996', 8587),
 ('1987', 3779),
 ('1986', 3668),
 ('1979', 3562),
 ('1974', 3530),
 ('1981', 4954),
 ('1983', 3279),
 ('1976', 2433),
 ('1975', 3200),
 ('1971', 1866),
 ('1970', 1597),
 ('1973', 2665)]